# LangChain 기초

LangChain을 사용한 LLM 애플리케이션 개발을 학습합니다.

## 학습 목표
1. LangChain 기본 구성요소 이해
2. 프롬프트 템플릿 사용
3. 체인 구성
4. 메모리 관리

In [ ]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

# OpenAI API 키 확인
api_key = os.getenv('OPENAI_API_KEY')
print(f"API Key configured: {'Yes' if api_key else 'No'}")

## 1. 기본 LLM 사용

In [ ]:
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI

# Chat 모델 초기화
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# 간단한 호출
response = llm.invoke("Python의 장점 3가지를 알려주세요.")
print(response.content)

## 2. 프롬프트 템플릿

In [ ]:
from langchain.prompts import PromptTemplate, ChatPromptTemplate

# 기본 프롬프트 템플릿
template = """{topic}에 대해 {style} 스타일로 설명해주세요.

설명은 {length}으로 해주세요.
"""

prompt = PromptTemplate(
    input_variables=["topic", "style", "length"],
    template=template
)

# 프롬프트 생성
formatted_prompt = prompt.format(
    topic="인공지능",
    style="어린이에게 설명하는",
    length="3문장 이내"
)
print(formatted_prompt)

In [ ]:
# Chat 프롬프트 템플릿
chat_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 친절하게 답변해주세요."),
    ("human", "{question}")
])

messages = chat_template.format_messages(
    role="Python 전문가",
    question="리스트와 튜플의 차이점은 무엇인가요?"
)

response = llm.invoke(messages)
print(response.content)

## 3. 체인 (Chains)

In [ ]:
from langchain.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser

# 간단한 체인
prompt = ChatPromptTemplate.from_template(
    "{product}의 창의적인 이름을 5개 제안해주세요."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"product": "친환경 텀블러"})
print(result)

In [ ]:
# 순차 체인
from langchain.chains import SequentialChain

# 첫 번째 체인: 제품 설명 생성
description_prompt = ChatPromptTemplate.from_template(
    "{product}에 대한 간단한 설명을 작성해주세요."
)
description_chain = description_prompt | llm | StrOutputParser()

# 두 번째 체인: 마케팅 문구 생성
marketing_prompt = ChatPromptTemplate.from_template(
    "다음 제품 설명을 바탕으로 마케팅 문구를 작성해주세요:\n\n{description}"
)
marketing_chain = marketing_prompt | llm | StrOutputParser()

# 실행
description = description_chain.invoke({"product": "스마트 물병"})
print("제품 설명:")
print(description)
print("\n마케팅 문구:")
marketing = marketing_chain.invoke({"description": description})
print(marketing)

## 4. 메모리 (대화 기록 유지)

In [ ]:
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain.chains import ConversationChain

# 버퍼 메모리
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# 대화
print(conversation.predict(input="안녕하세요! 제 이름은 김철수입니다."))
print("\n" + "="*50 + "\n")
print(conversation.predict(input="제 이름이 뭐라고 했죠?"))

In [ ]:
# 메모리 내용 확인
print("메모리 내용:")
print(memory.buffer)

## 5. 출력 파서

In [ ]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 출력 스키마 정의
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    rating: int = Field(description="평점 (1-10)")
    pros: List[str] = Field(description="장점 리스트")
    cons: List[str] = Field(description="단점 리스트")

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_template(
    """영화 '{movie}'에 대한 리뷰를 작성해주세요.

{format_instructions}
"""
)

chain = prompt | llm | parser

result = chain.invoke({
    "movie": "인셉션",
    "format_instructions": parser.get_format_instructions()
})

print(f"제목: {result.title}")
print(f"평점: {result.rating}/10")
print(f"장점: {result.pros}")
print(f"단점: {result.cons}")

## 연습 문제

1. 다양한 역할을 수행하는 챗봇을 만들어보세요.
2. 요약 메모리를 사용해 긴 대화를 관리해보세요.
3. 여러 체인을 연결해 복잡한 워크플로우를 구성해보세요.